In [2]:
import os
from ase.io import Trajectory, write

# List of input directories (one for each system)
input_dirs = [
    # "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/20ns_solvent_solute_0.5M/323_2K/md_omol_napf6_dme_re1",
    # "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/20ns_solvent_solute_0.5M/298_2K/md_omol_napf6_dme_re1",
    # "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/20ns_solvent_solute_0.5M/323_2K/md_omol_lipf6_pfactor_0.1_1fs_mask_t",
    # "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/20ns_solvent_0_1M/md_omol_naotf_dme_s1p1_omol"
    "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/20ns_solvent_0_1M/md_omol_napf6_dme_re1"
]

# The new output "root" directory for nvt_unstable
output_base = "/global/homes/y/yuejian/project/MLFF-distill/yuejian/electrolyte_application/ablate_distillation/ablation_diffusivity/nvt_patches_naotf_0_1M"

os.makedirs(output_base, exist_ok=True)
def get_common_path_prefix(dirs):
    segs = [d.split(os.path.sep) for d in dirs]
    prefix = []
    for tup in zip(*segs):
        if all(x == tup[0] for x in tup):
            prefix.append(tup[0])
        else:
            break
    return os.path.sep.join(prefix)

# get the parent up to .../ablate_distillation/ablation_diffusivity
common_prefix = get_common_path_prefix(input_dirs)
# Find just up to ablation_diffusivity
for test_dir in input_dirs:
    toks = test_dir.split(os.path.sep)
    if "ablation_diffusivity" in toks:
        idx = toks.index("ablation_diffusivity")
        ablation_diffusivity_prefix = os.path.sep.join(toks[:idx+1])
        break

common_prefix = ablation_diffusivity_prefix

for in_dir in input_dirs:
    # get the relative path from ablation_diffusivity onward
    rel_path = os.path.relpath(in_dir, common_prefix)
    out_dir = os.path.join(output_base, rel_path)
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    base_dir_name = os.path.basename(in_dir)
    in_traj = os.path.join(in_dir, f"{base_dir_name}.traj")
    out_traj = os.path.join(out_dir, f"{base_dir_name}.traj")
    out_log = os.path.join(out_dir, f"{base_dir_name}.log")

    # Read the last frame of the trajectory, but do not change or overwrite the original file
    with Trajectory(in_traj, "r") as traj:
        last_frame = traj[1000000]
    # Write only last frame to destination .traj (leave original untouched)
    write(out_traj, last_frame)

    # Prepare a blank .log file, or copy if needed. Here, just create an empty log file.
    with open(out_log, "w") as logf:
        logf.write("")

